# Linguistic Feature Clustering

This notebook computes correlations among linguistic features, performs hierarchical clustering, and exports feature-cluster assignments, dendrogram leaf order, and visualizations. Select the **normal** or **filtered** input using `FHS_FEATURE_VARIANT`.

In [ ]:
# Libraries required for hierarchical clustering and dendrogram visualizations.
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
from scipy.stats import spearmanr
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster
from scipy.spatial.distance import squareform
from pycirclize import Circos
from pycirclize.utils import ColorCycler


In [ ]:
# Match the directory and variant conventions used by the upstream analyses.
results_dir = Path(os.getenv("FHS_RESULTS_DIR", "Results"))
data_dir = Path(os.getenv("FHS_OUTPUT_DIR", str(results_dir / "Data")))
feature_variant = os.getenv("FHS_FEATURE_VARIANT", "normal").lower()
if feature_variant not in ("normal", "filtered"):
    raise ValueError("FHS_FEATURE_VARIANT must be 'normal' or 'filtered'.")

input_feature_filename = (
    "feature_np_tau_amyloid_filtered_20251116.csv"
    if feature_variant == "filtered"
    else "feature_np_tau_amyloid_20251116.csv"
)
figure_dir = results_dir / (
    "Figure_Results_filtered" if feature_variant == "filtered" else "Figure_Results"
)
table_dir = results_dir / (
    "Table_Results_filtered" if feature_variant == "filtered" else "Table_Results"
)
figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

# Preserve the original normal selection (zero-based columns 62:116).
# The filtered table has two additional feature columns; include both here.
# Confirm column positions against the linked datasets before reproducing results.
feature_column_stop = 118 if feature_variant == "filtered" else 116


## Load features

Keep the original row exclusion, selected feature-column positions, and mean imputation. The row selection is positional and assumes the same input row ordering used in the original study.

In [ ]:
whole_df = pd.read_csv(data_dir / input_feature_filename)

# Original sample selection: retain the first row and rows at zero-based positions 2–238.
whole_df = whole_df.iloc[[0] + list(range(2, 239))]

# Cluster the same feature block as the original notebook in normal mode.
X = whole_df.iloc[:, list(range(62, feature_column_stop))]
X = X.apply(lambda col: col.fillna(col.mean()) if col.dtype != 'object' else col)


## Spearman correlations and hierarchical clustering

Use the absolute Spearman correlation to define dissimilarities, then apply SciPy's Ward linkage exactly as in the original analysis.

In [ ]:
# Spearman correlation between features; correlation signs are ignored for clustering.
corr = spearmanr(X).correlation
corr = (corr + corr.T) / 2
np.fill_diagonal(corr, 1)

distance_matrix = 1 - np.abs(corr)
dist_linkage = hierarchy.ward(squareform(distance_matrix))


## Dendrograms and correlation heatmap

In [ ]:
# Standard dendrogram and corresponding feature-correlation heatmap.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8))
corr = spearmanr(X).correlation

# Ensure the correlation matrix is symmetric
corr = (corr + corr.T) / 2
np.fill_diagonal(corr, 1)

# We convert the correlation matrix to a distance matrix before performing
# hierarchical clustering using Ward's linkage.
distance_matrix = 1 - np.abs(corr)
dist_linkage = hierarchy.ward(squareform(distance_matrix))
dendro = hierarchy.dendrogram(
    dist_linkage, labels=X.columns.to_list(), ax=ax1, leaf_rotation=90,distance_sort=True,color_threshold=1.6
)
dendro_idx = np.arange(0, len(dendro["ivl"]))

ax2.imshow(corr[dendro["leaves"], :][:, dendro["leaves"]])
ax2.set_xticks(dendro_idx)
ax2.set_yticks(dendro_idx)
ax2.set_xticklabels(dendro["ivl"], rotation="vertical")
ax2.set_yticklabels(dendro["ivl"])
_ = fig.tight_layout()


In [ ]:
# Annotate linkage heights on a left-oriented dendrogram.
def augmented_dendrogram(*args, **kwargs):
    # Set the orientation to 'left' (rotated 90 degrees counterclockwise)
    kwargs['orientation'] = 'left'
    ddata = hierarchy.dendrogram(*args, **kwargs)

    if not kwargs.get('no_plot', False):
        for i, d in zip(ddata['icoord'], ddata['dcoord']):
            # For 'left' orientation, swap x and y meaning
            x = d[1]  # height is now on the x-axis
            y = 0.5 * sum(i[1:3])
            plt.plot(x, y, 'ro')
            plt.annotate("%.3g" % x, (x, y), xytext=(-25, -6),
                         textcoords='offset points',
                         va='center', ha='left', fontsize=6)

        # Rotate the y-axis tick labels to horizontal
        plt.yticks(rotation=0)

        # Add padding between y-axis ticks and the axis
        plt.gca().yaxis.set_tick_params(pad=10)

        # Adjust left margin to avoid label clipping
        plt.subplots_adjust(left=0.1)  # increase as needed (0.2–0.4)

    return ddata


In [ ]:
# Export the original PNG filename and image resolution.
plt.figure(1, figsize=(12, 12))
ddata = augmented_dendrogram(dist_linkage, labels=X.columns.to_list(), leaf_rotation=90, distance_sort=True,color_threshold=1.7)
plt.xticks(fontsize=8)   
plt.savefig(figure_dir / "feature_np_dendrogram.png", dpi=300, bbox_inches='tight') 
plt.show()


In [ ]:
# Optional Matplotlib polar dendrogram; display only, as in the original notebook.
def plot_circular_dendrogram(Z, labels, figsize=(10,10), leaf_fontsize=7,
                             start_angle=90, clockwise=True, color_threshold=None,
                             show_labels=True):
    """
    Z: linkage matrix (e.g. dist_linkage)
    labels: list of leaf labels (e.g. X.columns.tolist())
    """
    d = hierarchy.dendrogram(
        Z, labels=labels, no_plot=True, color_threshold=color_threshold
    )

    icoord = np.array(d["icoord"])
    dcoord = np.array(d["dcoord"])
    colors = d["color_list"]
    ivl = d["ivl"]                

    x_min, x_max = icoord.min(), icoord.max()
    theta = (icoord - x_min) / (x_max - x_min) * 2 * np.pi

    y_max = dcoord.max()
    r = np.log1p(y_max - dcoord)
    r = r / r.max() * y_max
    r = r + 1 * y_max

    theta = theta + np.deg2rad(start_angle)
    if clockwise:
        theta = 2*np.pi - theta

    fig = plt.figure(figsize=figsize)
    ax = plt.subplot(111, polar=True)
    ax.set_axis_off()
    
    for th, rr, c in zip(theta, r, colors):
        lw = 3.0 if rr.max() > 0.85 * r.max() else 1.8
        ax.plot(th, rr, color=c, linewidth=lw, solid_capstyle="round")
        
    if show_labels:
        n = len(ivl)
        leaf_x = np.array([5 + 10*i for i in range(n)], dtype=float)
        leaf_theta = (leaf_x - x_min) / (x_max - x_min) * 2*np.pi
        leaf_theta = leaf_theta + np.deg2rad(start_angle)
        if clockwise:
            leaf_theta = 2*np.pi - leaf_theta

        leaf_r = np.full(n, r.max() + 0.08*y_max)

        for t, rr, lab in zip(leaf_theta, leaf_r, ivl):
            ang = np.rad2deg(t)
            if 90 < ang < 270:
                rotation = ang + 180
                ha = "right"
            else:
                rotation = ang
                ha = "left"
            ax.text(
                t, rr, lab,
                rotation=rotation,
                rotation_mode="anchor",
                ha=ha,
                va="center",
                fontsize=leaf_fontsize + 1,
                fontweight="bold",
                path_effects=[pe.withStroke(linewidth=1.2, foreground="white")]
            )

    return fig, ax, d

fig, ax, d = plot_circular_dendrogram(
    dist_linkage,
    labels=X.columns.tolist(),
    figsize=(12,12),
    leaf_fontsize=7,
    start_angle=90,
    clockwise=True,
    color_threshold=1.72,   
    show_labels=True
)

# Optional circular plot export (disabled in the original notebook):
# plt.savefig(figure_dir / "feature_np_dendrogram_circular.png",
#             dpi=300, bbox_inches="tight")
plt.show()


## Cluster assignments and ordered leaf labels

Cut the same linkage matrix at 1.6, 1.7 and 1.8. Both CSVs are consumed by downstream visualizations; do not change their filenames.

In [ ]:
# Preserve the original distance thresholds and cluster labels.
threshold = 1.6
cluster_labels = fcluster(dist_linkage, t = threshold, criterion = 'distance')
feature_clusters_16 = pd.DataFrame({'Feature': X.columns, 'Cluster_16': cluster_labels})

threshold = 1.7
cluster_labels = fcluster(dist_linkage, t = threshold, criterion = 'distance')
feature_clusters_17 = pd.DataFrame({'Feature': X.columns, 'Cluster_17': cluster_labels})

threshold = 1.8
cluster_labels = fcluster(dist_linkage, t = threshold, criterion = 'distance')
feature_clusters_18 = pd.DataFrame({'Feature': X.columns, 'Cluster_18': cluster_labels})

feature_clusters= pd.merge(feature_clusters_16, feature_clusters_17,on = "Feature")
feature_clusters = pd.merge(feature_clusters,feature_clusters_18,on = 'Feature')


In [ ]:
# Preserve the output names expected by combined_plot.Rmd.
feature_clusters.to_csv(
    table_dir / "Linguistic_feature_dendrogram_cluster.csv", index=False
)

# Leaf order comes from the dendrogram with distance_sort=True above.
leaf_order = dendro["leaves"]
ordered_labels = [X.columns[i] for i in leaf_order]
ordered_labels_df = pd.DataFrame(ordered_labels, columns=["Label"])
ordered_labels_df.to_csv(
    table_dir / "Linguistic_feature_dendrogram_leaf_label.csv", index=False
)


## Newick tree and circular cluster visualization

Export the tree in Newick format, then color branches using the previously generated `Cluster_17` assignments. The original TIFF filename, including its spelling, is retained.

In [ ]:
def linkage_to_newick(Z, labels=None, precision=6):
    """
    Convert a SciPy linkage matrix Z to a Newick string.

    Parameters
    ----------
    Z : (n-1, 4) array
        SciPy linkage matrix.
    labels : list[str] or None
        Leaf labels of length n. If None, use "0..n-1".
    precision : int
        Decimal places for branch lengths in Newick.

    Returns
    -------
    newick : str
        Newick formatted tree string ending with ';'.
    """
    Z = np.asarray(Z)
    n = Z.shape[0] + 1

    if labels is None:
        labels = [str(i) for i in range(n)]
    if len(labels) != n:
        raise ValueError(f"labels length must be {n}, got {len(labels)}")

    # Height (distance) for each node id: leaves 0..n-1 have height 0
    height = np.zeros(2 * n - 1, dtype=float)
    for i in range(n - 1):
        node_id = n + i
        height[node_id] = float(Z[i, 2])

    def fmt(x):
        return f"{x:.{precision}f}"

    def build(node_id, parent_height):
        """Return (newick_subtree_str, branch_length_to_parent)."""
        if node_id < n:
            # leaf
            branch_len = max(parent_height - height[node_id], 0.0)
            return f"{labels[node_id]}:{fmt(branch_len)}"
        else:
            i = node_id - n
            left = int(Z[i, 0])
            right = int(Z[i, 1])
            node_h = height[node_id]

            # children subtrees with branch length = node_h - child_h
            left_str = build(left, node_h)
            right_str = build(right, node_h)

            branch_len = max(parent_height - node_h, 0.0)
            return f"({left_str},{right_str}):{fmt(branch_len)}"

    root_id = 2 * n - 2
    root_h = height[root_id]

    # Root has no parent; set parent_height=root_h so root branch length becomes 0
    newick = build(root_id, root_h) + ";"
    return newick


In [ ]:
newick_str = linkage_to_newick(dist_linkage, labels=X.columns.tolist(), precision=6)
out_nwk = figure_dir / "feature_tree.nwk"
with open(out_nwk, "w") as f:
    f.write(newick_str)


In [ ]:
# Build the circular tree using the Newick file created above.
circos, tv = Circos.initialize_from_tree(
    str(out_nwk),
    r_lim=(30, 100),
    leaf_label_size=8,
    line_kws=dict(color="lightgrey", lw=1.0),
)


In [ ]:
# Apply cluster labels from the table generated in this notebook.
df = pd.read_csv(table_dir / "Linguistic_feature_dendrogram_cluster.csv")
cluster2features = (
    df.groupby("Cluster_17")["Feature"]
      .apply(list)
      .to_dict()
)
group_name2feature_list = {f"Cluster_{k}": v for k, v in cluster2features.items()}


In [ ]:
# Set tree line color & label color
ColorCycler.set_cmap("tab10")
group_name2color = {name: ColorCycler() for name in group_name2feature_list.keys()}
for group_name, species_list in group_name2feature_list.items():
    color = group_name2color[group_name]
    tv.set_node_line_props(species_list, color=color, apply_label_color=True)

# Plot figure & set legend on center
fig = circos.plotfig()
_ = circos.ax.legend(
    handles=[Line2D([], [], label=n, color=c) for n, c in group_name2color.items()],
    labelcolor=group_name2color.values(),
    fontsize=8,
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
)
fig.savefig(
    figure_dir / "dandrogram_circle_1200.tif",
    dpi=1200,
    format="tiff",
    bbox_inches="tight"
)
